# Práctica 4A: Detección de plagio

## Realizado por: Eduardo Alanís García


## Instrucciones

En esta actividad, detectar, a nivel básico, por medio de una medida de similitud, el plagio que hay dentro de un conjunto de documentos.

Los datos son documentos de noticias adaptados a la tarea de detección de plagio. En el conjunto de datos del directorio “suspicious-documents”, hay documentos que tienen fragmentos plagiados de los documentos fuente “source-documents”, sin embargo, también hay otros documentos que simularon plagio pero no se plagió ningún párrafo de los documentos fuente.

La tarea es encontrar el nivel de plagio entre los archivos del directorio “source-documents” y los archivos del directorio “suspicious-documents”.

1. Crea un notebook para obtener la similitud entre dos documentos. Toma los documentos `source-document0001.txt` al `source-document0010.txt` de source-documents y por cada uno, encuentra los 10 archivos de suspicious-documents más parecidos. Indica los nombres y la similitud obtenida.
2. Aplica los preprocesamientos: usa solo letras minúsculas, quita los siguientes caracteres “;:,.\\-"‘/()[]¿?¡!{}\~<>|«»-—’”, remueve stopwords y aplica un proceso de stemming para reducir el vocabulario.

Nota: al usar la lista de stopwords para eliminarlas, si se aplicó stemming, aplicar stemming también a las stopwords debido a que alguna palabra podrían cambiar.

Usa el stemmer de NLTK Porter (inglés). [http://www.nltk.org/howto/stem.html](http://www.nltk.org/howto/stem.html)

Básicamente es usar:


```
from nltk.stem.porter import *
stemmer = PorterStemmer()
word = stemmer.stem('pages')
```

word contendrá la palabra normalizada por medio de stemming (‘page’). Aplicar ese proceso a todos los textos.

3. Para procesar la medida de similitud usar las medidas Jaccard y Dice
4. Contesta la evaluación de esta práctica en el Aula Virtual.

Los datos son archivos de texto plano UTF-8 en inglés:

- source-documents.zip.
- suspicious-documents.zip.

### Carga de datos



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import glob

direccion = "/content/drive/MyDrive/MCDI-INFOTEC/1. Primer semestre/Manejo y procesamiento de la informacion/Unidad 4/4A. Práctica Detección de plagio/"

direccion_source = os.path.join(direccion, "source-documents")
direccion_suspicious = os.path.join(direccion, "suspicious-documents")

rutas_source = sorted(glob.glob(os.path.join(direccion_source, "*.txt")))
rutas_suspicious = sorted(glob.glob(os.path.join(direccion_suspicious, "*.txt")))

print("Documentos fuente disponibles:", len(rutas_source))
print("Documentos sospechosos disponibles:", len(rutas_suspicious))

Documentos fuente disponibles: 237
Documentos sospechosos disponibles: 2370


### Instalar e importar NLTK

In [ ]:
%pip install nltk


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

nltk.download("stopwords")

stemmer = PorterStemmer()
stopwords_ingles = stopwords.words("english")

print("Stopwords en inglés:", len(stopwords_ingles))


Stopwords en inglés: 198


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


### Preprocesar los documentos

Aplica los preprocesamientos: usa solo letras minúsculas, quita los siguientes caracteres “;:,.\-"‘/()[]¿?¡!{}~<>|«»-—’”, remueve stopwords y aplica un proceso de stemming para reducir el vocabulario.


In [ ]:
caracteres_eliminar = ";:,.-\"‘/()[]¿?¡!{}~<>|«»-—’"

stopwords_stem = set()

for palabra in stopwords_ingles:
  stopwords_stem.add(stemmer.stem(palabra))


def leer_archivo(ruta):
  with open(ruta, "r", encoding="utf-8") as archivo:
    texto = archivo.read()

  return texto


def preprocesar_texto(texto):
  texto = texto.lower()

  for caracter in caracteres_eliminar:
    texto = texto.replace(caracter, "")

  palabras = texto.split()
  palabras_procesadas = []

  for palabra in palabras:
    palabra_stem = stemmer.stem(palabra)

    if palabra_stem not in stopwords_stem:
      palabras_procesadas.append(palabra_stem)

  return set(palabras_procesadas)


In [ ]:
nombres_fuente = []
for numero in range(1, 11):
  nombres_fuente.append(f"source-document{numero:04d}.txt")

print(nombres_fuente)

rutas_source_por_nombre = {}
for ruta in rutas_source:
  nombre_archivo = os.path.basename(ruta)
  rutas_source_por_nombre[nombre_archivo] = ruta

nombres_faltantes = []
for nombre_archivo in nombres_fuente:
  if nombre_archivo not in rutas_source_por_nombre:
    nombres_faltantes.append(nombre_archivo)

if len(nombres_faltantes) > 0:
  raise FileNotFoundError("Faltan documentos")


documentos_fuente = {}
for nombre_archivo in nombres_fuente:
  texto = leer_archivo(rutas_source_por_nombre[nombre_archivo])
  documentos_fuente[nombre_archivo] = preprocesar_texto(texto)


documentos_sospechosos = {}
for ruta in rutas_suspicious:
  nombre_archivo = os.path.basename(ruta)
  texto = leer_archivo(ruta)
  documentos_sospechosos[nombre_archivo] = preprocesar_texto(texto)

print("Documentos fuente que se compararán:", len(documentos_fuente))
print("Documentos sospechosos que se compararán:", len(documentos_sospechosos))


['source-document0001.txt', 'source-document0002.txt', 'source-document0003.txt', 'source-document0004.txt', 'source-document0005.txt', 'source-document0006.txt', 'source-document0007.txt', 'source-document0008.txt', 'source-document0009.txt', 'source-document0010.txt']
Documentos fuente que se compararán: 10
Documentos sospechosos que se compararán: 2370


### Definir las medidas de similitud

Para dos conjuntos de palabras $A$ y $B$:

- **Jaccard:** $|A \cap B| / |A \cup B|$
- **Dice:** $2|A \cap B| / (|A| + |B|)$



In [ ]:
def calcular_jaccard(documento_1, documento_2):
  interseccion = documento_1.intersection(documento_2)
  union = documento_1.union(documento_2)

  if len(union) == 0:
    return 0

  return len(interseccion) / len(union)


def calcular_dice(documento_1, documento_2):
  interseccion = documento_1.intersection(documento_2)
  total_palabras = len(documento_1) + len(documento_2)

  if total_palabras == 0:
    return 0

  return (2 * len(interseccion)) / total_palabras


### Obtener los 10 documentos más parecidos



In [ ]:
resultados_jaccard = {}
resultados_dice = {}

for nombre_fuente in nombres_fuente:
  similitudes_jaccard = []
  similitudes_dice = []
  documento_fuente = documentos_fuente[nombre_fuente]

  for nombre_sospechoso in documentos_sospechosos:
    documento_sospechoso = documentos_sospechosos[nombre_sospechoso]

    similitud_jaccard = calcular_jaccard(documento_fuente, documento_sospechoso)
    similitud_dice = calcular_dice(documento_fuente, documento_sospechoso)

    similitudes_jaccard.append((nombre_sospechoso, similitud_jaccard))
    similitudes_dice.append((nombre_sospechoso, similitud_dice))

  similitudes_jaccard.sort(key=lambda resultado: resultado[1], reverse=True)
  similitudes_dice.sort(key=lambda resultado: resultado[1], reverse=True)

  resultados_jaccard[nombre_fuente] = similitudes_jaccard[:10]
  resultados_dice[nombre_fuente] = similitudes_dice[:10]




In [ ]:
for nombre_fuente in nombres_fuente:
  print("\nDocumento fuente:", nombre_fuente)

  print("Top 10 con Jaccard")
  for nombre_sospechoso, similitud in resultados_jaccard[nombre_fuente]:
    print(f"  {nombre_sospechoso}: {similitud:.4f} ({similitud * 100:.2f}%)")

  print("Top 10 con Dice")
  for nombre_sospechoso, similitud in resultados_dice[nombre_fuente]:
    print(f"  {nombre_sospechoso}: {similitud:.4f} ({similitud * 100:.2f}%)")



Documento fuente: source-document0001.txt
Top 10 con Jaccard
  suspicious-document0010.txt: 0.1727 (17.27%)
  suspicious-document2205.txt: 0.1561 (15.61%)
  suspicious-document2048.txt: 0.1551 (15.51%)
  suspicious-document2121.txt: 0.1544 (15.44%)
  suspicious-document1267.txt: 0.1521 (15.21%)
  suspicious-document2167.txt: 0.1511 (15.11%)
  suspicious-document1709.txt: 0.1508 (15.08%)
  suspicious-document1402.txt: 0.1507 (15.07%)
  suspicious-document2025.txt: 0.1507 (15.07%)
  suspicious-document0008.txt: 0.1504 (15.04%)
Top 10 con Dice
  suspicious-document0010.txt: 0.2946 (29.46%)
  suspicious-document2205.txt: 0.2701 (27.01%)
  suspicious-document2048.txt: 0.2685 (26.85%)
  suspicious-document2121.txt: 0.2674 (26.74%)
  suspicious-document1267.txt: 0.2640 (26.40%)
  suspicious-document2167.txt: 0.2625 (26.25%)
  suspicious-document1709.txt: 0.2621 (26.21%)
  suspicious-document1402.txt: 0.2620 (26.20%)
  suspicious-document2025.txt: 0.2620 (26.20%)
  suspicious-document0008.txt

### Contesta la evaluación de esta práctica en el Aula Virtual.